# shelf-vision: 11m fine-tune on pseudo-labelled train, at 1280

`configs/detect/yolo11m-sku110k-pl.yaml`: starts from the 11m run's weights, trains
25 epochs at imgsz 1280 on SKU-110K train with ~26k extra boxes that the 11m model
found where the original annotators left shelves unlabelled. Val/test labels are the
originals, so the test numbers compare like for like with earlier runs.

**Before running:**
- Settings → Accelerator **GPU T4 x2**, Internet **on**.
- Add Input → Notebooks → the 11m training notebook (output has
  `runs/detect/yolo11m-sku110k-*h/weights/best.pt`).
- Add Input → Datasets → `sku110k-pseudo-labels` (`train_*.txt`).
- **Save Version → Save & Run All (Commit)**.

In [ ]:
import glob
import os
import shutil

DATA_DIR = "/tmp/shelf-data"  # ~24 GB peak; /kaggle/working is limited to ~20 GB
os.environ["SHELF_DATA_DIR"] = DATA_DIR
CFG = "shelf-vision/configs/detect/yolo11m-sku110k-pl.yaml"

found = sorted(glob.glob("/kaggle/input/**/yolo11m-sku110k-*h/weights/best.pt", recursive=True))
assert found, "attach the 11m training notebook's output as an input (see above)"
BASE = found[-1]
print("starting from", BASE)

pl = sorted(glob.glob("/kaggle/input/**/sku110k-pseudo-labels/**/train_*.txt", recursive=True))
assert len(pl) > 8000, f"attach the pseudo-label dataset (found {len(pl)} files)"
print(len(pl), "pseudo-labelled train files")

!nvidia-smi --query-gpu=name,memory.total --format=csv
free_gb = shutil.disk_usage("/tmp").free / 2**30
assert free_gb > 30, f"only {free_gb:.0f} GB free in /tmp, need ~30 GB for the dataset"

In [ ]:
BRANCH = "main"
!git clone -q -b {BRANCH} https://github.com/rsuvbe/shelf-vision
!git -C shelf-vision log --oneline -1
!pip install -q -e shelf-vision
!shelf doctor

## Data

Download as usual, then overlay the pseudo-labelled **train** labels only.

In [ ]:
!bash shelf-vision/scripts/fetch_sku110k.sh {DATA_DIR}
!shelf prepare {CFG}
!rm -f {DATA_DIR}/SKU110K_fixed.tar.gz

lbl_dir = f"{DATA_DIR}/SKU-110K/labels"
assert os.path.isdir(lbl_dir), lbl_dir
before = sum(1 for _ in open(f"{lbl_dir}/train_6216.txt"))
for p in pl:
    shutil.copyfile(p, os.path.join(lbl_dir, os.path.basename(p)))
for c in glob.glob(f"{DATA_DIR}/SKU-110K/*.cache"):
    os.remove(c)  # force ultralytics to re-read labels
after = sum(1 for _ in open(f"{lbl_dir}/train_6216.txt"))
print(f"train_6216: {before} -> {after} boxes")
assert after > before

## Fine-tune

In [ ]:
!shelf train {CFG} --weights {BASE} 2>&1 | tr '\r' '\n' | grep -v '━' | tee /kaggle/working/train.log

## Test set

Starting weights at their own 960 and at 1280, then the fine-tuned ones at 1280,
all in this session on the original test labels.

In [ ]:
FT = max(glob.glob("/kaggle/working/runs/detect/yolo11m-sku110k-pl*/weights/best.pt"))
BASE_CFG = "shelf-vision/configs/detect/yolo11m-sku110k.yaml"
# base@960 pins the size and IoU it was originally measured at; the base config
# now carries deployment settings (1280, IoU 0.8) in its infer section
for name, w, cfg, extra in [("base@960", BASE, BASE_CFG, "--imgsz 960 --iou 0.7"), ("base@1280", BASE, CFG, ""), ("pl@1280", FT, CFG, "")]:
    print(f"== {name}: {w}")
    !shelf evaluate {w} {cfg} --split test {extra} 2>&1 | tr '\r' '\n' | grep -v '━' | tail -3 | tee -a /kaggle/working/train.log

Output to download: `runs/detect/yolo11m-sku110k-pl/` (weights, `results.csv`, plots) and `train.log`.